# 82514 · Sesión S17 — Procesado clásico: de la foto de la mesa a la pieza en el robot

**Bloque 6** · lunes 16 de noviembre de 2026 · 2 h · IQS Universitat Ramon Llull

La foto de la mesa del puesto de picking: el umbral y la luz; la pieza, en milímetros y en ángulos; la convolución, Canny y las esquinas.

| | |
|---|---|
| **En clase** | Nueve minutos: la sección 0, sin deteneros, y las secciones 1 a 3, con el ejercicio de clase. Tened a mano vuestro JSON de S16. Al acabar, abrid el S17b y ejecutad su primera celda. |
| **En casa** | Sección 4 y ejercicios 1 a 3; las soluciones, al final. |
| **Entrega** | Al acabar la clase, individual: este cuaderno y el S17b, ejecutados y con «✎ Vuestra respuesta» escrita (Archivo → Descargar → Descargar .ipynb), a la Tarea «Taller S17: procesado de imagen» de Moodle. Sin nota: presentado o no presentado. |

Cada paso tiene tres celdas: **▶ Parámetros**, lo único que se cambia; **⚙ Cálculo**, que no hace falta tocar; y **✔ Resultado**, lo que sale.

---

In [ ]:
# @title ⚙ Instalación · ejecutad esta celda primero
# Ejecutar una sola vez. En Colab tarda unos veinte segundos.
import importlib, subprocess, sys

def asegurar(mods):
    faltan = []
    for pip_name, import_name in mods:
        try:
            importlib.import_module(import_name)
        except ImportError:
            faltan.append(pip_name)
    if faltan:
        print('Instalando:', ' '.join(faltan))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + faltan, check=False)
    else:
        print('Todo instalado ya.')

asegurar([('numpy', 'numpy'), ('matplotlib', 'matplotlib')])

import importlib.metadata as md

OPENCV = '4.14.0.94'          # la versión de todo el bloque 6 (cv2.__version__ = '4.14.0')
FIJAS = {'scikit-image': '0.25.2'}               # las de Colab: se comprueban y, si cambian, se instalan antes de importarlas

def opencv_activo():
    """Versión de OpenCV que se importaría, mirada en otro proceso: Colab trae tres paquetes de OpenCV que
    comparten la carpeta cv2, y lo que dice pip no basta para saber cuál se carga."""
    r = subprocess.run([sys.executable, '-c', 'import cv2; print(cv2.__version__)'], capture_output=True, text=True)
    return r.stdout.strip()

def fijar():
    if opencv_activo() != OPENCV.rsplit('.', 1)[0]:
        print('Instalando OpenCV', OPENCV, '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'opencv-python-headless=={OPENCV}'], check=False)
    for paquete, version in FIJAS.items():
        try:
            ya = md.version(paquete)
        except md.PackageNotFoundError:
            ya = None
        if ya != version:
            print('Instalando', paquete, version, '...')
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{paquete}=={version}'], check=False)

fijar()
import os, json, time, tempfile, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import cv2
warnings.filterwarnings('ignore')
EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.6)
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

def sin_ejes(*axs):
    for ax in axs:
        ax.set_xticks([]); ax.set_yticks([])

print('OpenCV', cv2.__version__, '· numpy', np.__version__, '· en Colab:', EN_COLAB)
if cv2.__version__ != OPENCV.rsplit('.', 1)[0]:
    print('AVISO: este cuaderno se probó con OpenCV', OPENCV, '. Entorno de ejecución → Reiniciar sesión, y vuelve a ejecutar esta celda.')
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

## 0. La mesa del puesto de picking

*En clase, sin deteneros · diapositivas 21 y 32*

La pieza, un disco de 40 mm en P = (0,4; 0,2) m, con la luz del techo y con una lámpara.
Poned `EQUIPO` y `LETRA` y ejecutad desde aquí; en Colab, subid vuestro JSON de S16.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# la cámara que hace las fotos: 0, la del curso; de 1 a 5, la unidad de vuestro equipo, la de S16
EQUIPO = 0  # @param {type:"number"}
# la letra de calibracion_E<n><letra>.json, el JSON de S16 de cualquiera de los dos (con EQUIPO = 0 no se usa)
# si sale el AVISO de la letra: corregid aquí EQUIPO y LETRA y volved a ejecutar desde esta celda
LETRA = 'A'  # @param ["A", "B", "C", "D", "E"]
# la lámpara: la luz de la mesa a la izquierda y a la derecha, en fracción de la del techo
LUZ_LAMPARA = (0.5, 1.0)  # @param {type:"raw"}
# hacia dónde está la lámpara, en metros de mesa por metro de altura: (1.73, 0.5), a 30° sobre la mesa, a la derecha
DIR_LAMPARA = (1.73, 0.5)  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo · el simulador: ejecutadlo sin abrirlo
# ── El simulador de la mesa del puesto de picking y de las cámaras de los equipos ──────────────────────────
# Ejecutadla sin deteneros: es la cámara de S16 apuntando a la mesa. Cada píxel (submuestreado 2 x 2) es un rayo que
# sale de la cámara con su K y su distorsión, se gira al mundo con la pose de la cámara y se corta con la mesa y con
# las piezas: la cara de arriba, a 10 mm; el lateral; y la sombra, si hay una lámpara. La mesa tiene la veta de la
# madera. Después, desenfoque, ruido del sensor (2 niveles) y 8 bits, como una foto. Se conoce la verdad píxel a
# píxel: qué píxeles son pieza y dónde está cada una.
W, H = 640, 480
F = 800.0
K_CURSO = np.array([[F, 0, W/2], [0, F, H/2], [0, 0, 1.0]])
DIST_CURSO = np.array([-0.25, 0.06, 0.0, 0.0, 0.0])
A1, A2 = 0.35, 0.20
TC = np.array([0.30, 0.20, 1.0])
R_0C = np.diag([1.0, -1.0, -1.0])            # Rx(180°): la camara mira hacia abajo
P_PIEZA = np.array([0.40, 0.20])
H_PIEZA, D_PIEZA = 0.010, 0.040

T_0C_NOMINAL = np.eye(4); T_0C_NOMINAL[:3, :3] = R_0C; T_0C_NOMINAL[:3, 3] = TC   # el plano del montaje


def T_C0_verdad():
    M = np.eye(4); M[:3, :3] = R_0C.T; M[:3, 3] = -R_0C.T @ TC
    return M


class Pieza:
    """Prisma recto de altura h: 'disco' (diametro d), 'cuadrado' (lado d) o 'triangulo' (lado d), girado ang."""
    def __init__(self, forma, x, y, d, ang=0.0, h=H_PIEZA, albedo=0.80):
        self.forma, self.c, self.d, self.ang, self.h, self.albedo = forma, np.array([x, y]), d, ang, h, albedo
        if forma == 'cuadrado':
            v = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]])*d/2
        elif forma == 'triangulo':
            r = d/np.sqrt(3)
            v = np.array([[r*np.cos(a), r*np.sin(a)] for a in (np.pi/2, np.pi/2 + 2*np.pi/3, np.pi/2 + 4*np.pi/3)])
        else:
            v = None
        if v is not None:
            ca, sa = np.cos(ang), np.sin(ang)
            self.v = v @ np.array([[ca, sa], [-sa, ca]]) + self.c

    def dentro(self, X, Y):
        if self.forma == 'disco':
            return (X - self.c[0])**2 + (Y - self.c[1])**2 <= (self.d/2)**2
        ok = np.ones(X.shape, bool)
        v = self.v
        for a, b in zip(v, np.roll(v, -1, axis=0)):      # poligono convexo, vertices en sentido antihorario
            ok &= (b[0] - a[0])*(Y - a[1]) - (b[1] - a[1])*(X - a[0]) >= 0
        return ok

    def area_px(self, f=F, Z=1.0):
        """Area aproximada de la cara de arriba en la imagen (a la distancia Z - h)."""
        e = f/(Z - self.h)
        if self.forma == 'disco':
            return np.pi*(self.d/2*e)**2
        if self.forma == 'cuadrado':
            return (self.d*e)**2
        return np.sqrt(3)/4*(self.d*e)**2


def vetas(X, Y):
    """La veta de la madera de la mesa: rayas onduladas a lo largo de y, cada 8 mm, de un 8 % de amplitud."""
    return 0.08*np.sin(2*np.pi*X/0.008 + 2.5*np.sin(2*np.pi*Y/0.15))


ESCENA = [Pieza('disco', 0.40, 0.20, 0.040),                 # la pieza: en P, el punto de la inversa de S10
          Pieza('cuadrado', 0.12, 0.34, 0.040, ang=0.35),     # dos piezas que no son la nuestra
          Pieza('triangulo', 0.56, 0.40, 0.050, ang=0.2)]


class Mesa:
    """La camara (K y distorsion de la unidad) a 1 m de la mesa, cenital: por defecto, en el plano del montaje,
    T_0C = Trans(0,3; 0,2; 1)·Rx(180°); con T_0C, donde de verdad la dejo su soporte (la unidad de S16)."""
    def __init__(self, K=K_CURSO, dist=DIST_CURSO, T_0C=None, ss=2):
        T = T_0C_NOMINAL if T_0C is None else np.asarray(T_0C, float)
        self.K, self.dist, self.ss = K, dist, ss
        self.R, self.o = T[:3, :3], T[:3, 3]                        # giro y origen de la camara en {0}
        us = (np.arange(W*ss) + 0.5)/ss - 0.5
        vs = (np.arange(H*ss) + 0.5)/ss - 0.5
        U, V = np.meshgrid(us, vs)
        pts = np.stack([U.ravel(), V.ravel()], -1).reshape(-1, 1, 2)
        n = cv2.undistortPointsIter(pts, K, dist, None, None,
                                    (cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 60, 1e-12)).reshape(-1, 2)
        d = (self.R @ np.vstack([n.T, np.ones(len(n))])).T        # direccion del rayo en el mundo
        self.d = d
        self.X0, self.Y0 = self.corta(0.0)                         # donde corta cada rayo a la mesa

    def corta(self, z):
        lam = (z - self.o[2])/self.d[:, 2]
        return self.o[0] + lam*self.d[:, 0], self.o[1] + lam*self.d[:, 1]

    def foto(self, piezas=ESCENA, rng=None, luz=(1.0, 1.0), mesa=0.35, lateral=0.30, sombra=0.45,
             lampara=None, blur=0.7, ruido=2.0, textura=vetas, mascara=False):
        """luz = (en x = -0,1 m, en x = 0,7 m): la iluminacion de la mesa varia en linea recta a lo ancho.
        lampara: hacia donde esta la lampara, en metros de mesa por metro de altura (None: luz del techo, sin
        sombras; (1,73; 0,5), una lampara a 30° sobre la mesa, a la derecha). Devuelve la imagen
        en gris (uint8) y, si mascara, la verdad: un entero por pixel (0 mesa, i+1 la cara de arriba de la pieza i)."""
        rng = np.random.default_rng(0) if rng is None else rng
        X0, Y0 = self.X0, self.Y0
        L = luz[0] + (luz[1] - luz[0])*np.clip((X0 + 0.10)/0.80, 0, 1)
        alb = np.full(X0.shape, mesa)
        if textura is not None:
            alb = alb*(1 + textura(X0, Y0))
        verdad = np.zeros(X0.shape, np.int32)
        for i, p in enumerate(piezas):
            # solo los rayos que pasan cerca de la pieza (mas rapido: la escena entera son 1,2 millones de rayos)
            alc = p.d + (0 if lampara is None else np.hypot(*lampara)*p.h) + 0.01
            j = np.flatnonzero((X0 - p.c[0])**2 + (Y0 - p.c[1])**2 < alc**2)
            x0, y0 = X0[j], Y0[j]
            base = p.dentro(x0, y0)
            if lampara is not None:      # sombra: desde ese punto de la mesa, la lampara se ve a traves de la pieza
                somb = np.zeros(len(j), bool)
                for z in (0.25*p.h, 0.5*p.h, 0.75*p.h, p.h):
                    somb |= p.dentro(x0 + lampara[0]*z, y0 + lampara[1]*z)
                alb[j] = np.where(somb & ~base, alb[j]*sombra, alb[j])
            lam = (p.h - self.o[2])/self.d[j, 2]
            top = p.dentro(self.o[0] + lam*self.d[j, 0], self.o[1] + lam*self.d[j, 1])
            alb[j] = np.where(base & ~top, lateral, alb[j])     # el lateral: entra por el borde y no sale por la cara
            alb[j] = np.where(top, p.albedo, alb[j])
            verdad[j] = np.where(top, i + 1, verdad[j])
        img = 255*alb*L
        ss = self.ss
        img = img.reshape(H*ss, W*ss).reshape(H, ss, W, ss).mean(axis=(1, 3))
        img = cv2.GaussianBlur(img, (0, 0), blur)
        img = img + rng.normal(0, ruido, img.shape)
        img = np.clip(np.round(img), 0, 255).astype(np.uint8)
        if mascara:     # la verdad en la rejilla de pixeles: la pieza que cubre al menos 2 de los 4 subpixeles
            vs = verdad.reshape(H, ss, W, ss).transpose(0, 2, 1, 3).reshape(H, W, ss*ss)
            v = np.zeros((H, W), np.int32)
            for i in range(1, len(piezas) + 1):
                v[(vs == i).sum(-1)*2 >= ss*ss] = i
            return img, v
        return img


def a_la_mesa(uv, K, dist, T_C0, z=0.0):
    """Del pixel al plano z de {0}: quitar la distorsion, rayo K^-1·p, girarlo al mundo y cortarlo con el plano."""
    uv = np.atleast_2d(np.asarray(uv, float)).reshape(-1, 1, 2)
    n = cv2.undistortPointsIter(uv, K, np.asarray(dist, float), None, None,
                                (cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 100, 1e-12)).reshape(-1, 2)
    n = np.column_stack([n, np.ones(len(n))])
    T = np.linalg.inv(np.asarray(T_C0, float))
    R, o = T[:3, :3], T[:3, 3]
    d = (R @ n.T).T
    lam = (z - o[2])/d[:, 2]
    return (o + lam[:, None]*d)[:, :2]


def ik2r(x, y, codo=+1):
    """La inversa cerrada del 2R de S10 (a1 = 0,35 m, a2 = 0,2 m); codo = +1, codo abajo."""
    c2 = (x*x + y*y - A1**2 - A2**2)/(2*A1*A2)
    q2 = codo*np.arccos(np.clip(c2, -1, 1))
    q1 = np.arctan2(y, x) - np.arctan2(A2*np.sin(q2), A1 + A2*np.cos(q2))
    return q1, q2


def piezas_en(mask, area_min=300, area_max=3000):
    """Componentes conexas de una mascara binaria: area, centroide (momentos), circularidad y caja."""
    n, et, st, cen = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = []
    for k in range(1, n):
        a = st[k, cv2.CC_STAT_AREA]
        if a < area_min or a > area_max:
            continue
        m = (et == k).astype(np.uint8)
        M = cv2.moments(m, binaryImage=True)
        cont = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[0][0]
        per = cv2.arcLength(cont, True)
        x, y, w, h = st[k, :4]
        out.append(dict(area=M['m00'], u=M['m10']/M['m00'], v=M['m01']/M['m00'],
                        circ=4*np.pi*M['m00']/per**2, caja=(x, y, x + w, y + h)))
    return out

import zlib, base64
D2R = np.pi/180

# Las unidades de S16, con su montaje: la misma tabla codificada que el simulador del cuaderno S16 (sim_celda.py).
# Es la verdad que se desvela el jueves 26, en S20: no la decodifiquéis.
_U = 'eNo9kFkOwyAMRO+Sb8fCC0t6laonqXr3zji0ijCE4MebvI92PJ6rNW0SzurJejb1LlgNlnvcj/7HSw5D81yhjuapKR5drZonT1wsrQV3MLuknCFEJrcCiwWKU8Fcl4RhBwqxKU5A3+150/Cli8tZCDpibergBDmeFSUrytJZHIqYybbYUiFdqMBBpw5EMtAVeA8baHa/IERE3T63iu0ZyXD7kPsIS4rVn+kEzcoUBk1PB5ShqWH+y7KVhgA5S6RSoczX5wuk8U/j'
_UNIDADES = {int(k): v for k, v in json.loads(zlib.decompress(base64.b64decode(_U))).items()}


def _Rx(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def _Ry(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def _Rz(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def unidad(e):
    """La camara que hace las fotos: K, distorsion y T_0C de la unidad e (0, la del curso, en el plano del montaje),
    construidos como Camara.__init__ del simulador de S16."""
    fx, cx, cy, k1, k2, p1, p2, dx, dy, dz, ax, ay, az = _UNIDADES[e]
    M = np.eye(4); M[:3, :3] = _Rz(az*D2R) @ _Ry(ay*D2R) @ _Rx(ax*D2R); M[:3, 3] = np.array([dx, dy, dz])*1e-3
    return np.array([[fx, 0, cx], [0, fx, cy], [0, 0, 1.0]]), np.array([k1, k2, p1, p2, 0.0]), T_0C_NOMINAL @ M

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
def _clave(d, *nombres):
    for n in nombres:
        if d.get(n) is not None:
            return d[n]
    return None

def calibracion(e, letra):
    """Con qué se mide: (K, distorsión, T_C0). La de la cámara del curso, exacta; o la de un JSON de S16, el de uno de los dos."""
    if e == 0:
        return K_CURSO, DIST_CURSO, T_C0_verdad()
    assert letra in list('ABCDE'), "LETRA es la del fichero de S16: una de 'A', 'B', 'C', 'D' o 'E', entre comillas"
    nombre = f'calibracion_E{e}{letra}.json'
    if not os.path.exists(nombre) and EN_COLAB:
        from google.colab import files
        print('Sube', nombre, '(el que guardó en S16 uno de los dos; también está en su entrega de Moodle):')
        subidos = files.upload()
        if subidos:
            nombre = next(iter(subidos))          # el navegador puede haberlo llamado «… (1).json»
    if os.path.exists(nombre):
        d = json.load(open(nombre, encoding='utf-8'))
        ident = f"E{d.get('equipo', e)}{str(d.get('pareja', letra)).strip().upper()}"   # 'pareja' guarda la letra
        if ident != f'E{e}{letra}':
            print(f'AVISO: {nombre} es la calibración de {ident}, no la de E{e}{letra}: poned en EQUIPO y LETRA '
                  'los de ese fichero en ▶ Parámetros (sección 0) y volved a ejecutar desde allí.')
        K = np.array(_clave(d, 'K', 'matriz_K'), float)
        dist = np.array(_clave(d, 'distorsion_k1_k2_p1_p2_k3', 'dist', 'distorsion'), float).ravel()
        T = np.array(_clave(d, 'T_C0', 'extrinsecos'), float)
        if K.shape == (3, 3) and dist.size in (4, 5, 8, 12, 14) and T.shape == (4, 4) and np.isfinite(dist).all():
            rms = _clave(d, 'rms_px', 'rms')
            print(f'Calibración de {ident} ({nombre}): fx = {coma(K[0, 0], 1)} px, RMS = {coma(rms, 3) if rms is not None else "?"} px')
            return K, dist, T
        print('AVISO:', nombre, 'no trae K (3 x 3), la distorsión y T_C0 (4 x 4).')
    print('AVISO: sin calibración: se mide con la hoja de datos (800 px, sin distorsión) y el plano del montaje.')
    return K_CURSO, np.zeros(5), T_C0_verdad()

cam = Mesa(*unidad(EQUIPO))                   # la cámara que hace las fotos (su verdad no se imprime)
CAL = calibracion(EQUIPO, LETRA)              # con qué se mide: (K, distorsión, T_C0); en Colab, pide el JSON
LAMPARA = dict(luz=LUZ_LAMPARA, lampara=DIR_LAMPARA)
I_t, V_t = cam.foto(mascara=True)                                  # la luz del techo
I_l, V_l = cam.foto(rng=np.random.default_rng(1), mascara=True, **LAMPARA)   # con la lámpara
CON = 'la cámara del curso' if EQUIPO == 0 else ('vuestra calibración' if CAL[0] is not K_CURSO else 'la hoja de datos (no hay calibración)')
# para las figuras: las dos fotos y su verdad (las secciones 3 y 4 cambian I_t, I_l, V_t y V_l si EQUIPO no es 0) y la fila del centro del disco
FOTOS_0, VERDAD_0 = (I_t, I_l), (V_t, V_l)
FILA_P = int(np.argwhere(V_t == 1)[:, 0].mean().round())       # 240 con la cámara del curso
U_P = np.flatnonzero(V_t[FILA_P] == 1)                         # las columnas del disco en esa fila
LADO_0 = -1 if DIR_LAMPARA[0] > 0 else 1                       # dónde cae la sombra, al otro lado de la lámpara: -1, a la izquierda

In [ ]:
# @title ✔ Resultado
print('Las fotos, con', 'la cámara del curso' if EQUIPO == 0 else f'la unidad del equipo {EQUIPO}',
      f'· se mide con {CON} (f_x = {coma(CAL[0][0, 0], 1)} px)')
if EQUIPO == 0:
    print('Con EQUIPO = 0 no se pide ningún JSON. Para medir con vuestra calibración de S16, poned EQUIPO (1 a 5) y LETRA\n'
          'en ▶ Parámetros y volved a ejecutar desde allí: la celda ⚙ de abajo os pedirá el fichero.')
fig, axs = plt.subplots(1, 3, figsize=(15, 3.9), gridspec_kw=dict(width_ratios=[1, 1, 1.25], left=0.01, right=0.99, bottom=0.13, top=0.9, wspace=0.12))
for ax, I_, t in zip(axs, FOTOS_0, ('La luz del techo', 'Con una lámpara a la ' + ('derecha' if LADO_0 < 0 else 'izquierda'))):
    ax.imshow(I_, cmap='gray', vmin=0, vmax=255); ax.axhline(FILA_P, color='w', lw=1, ls='--')
    ax.set_title(t, fontsize=12); sin_ejes(ax)
ax = axs[2]
ax.plot(FOTOS_0[0][FILA_P], color=IQS_AZUL, lw=1.2, label='la luz del techo')
ax.plot(FOTOS_0[1][FILA_P], color=AMBAR, lw=1.2, label='con la lámpara')
ax.text(U_P.mean(), FOTOS_0[0][FILA_P, U_P].max() + 6, 'el disco', ha='center', va='bottom', fontsize=10)
ini_ = U_P[0] - 20 if LADO_0 < 0 else U_P[-1] + 1              # los 20 px junto al disco, al otro lado de la lámpara
u_s = ini_ + int(np.argmin(FOTOS_0[1][FILA_P, ini_:ini_ + 20]))   # el fondo de la sombra
ax.annotate('la sombra', (u_s, FOTOS_0[1][FILA_P, u_s]), (u_s + 70*LADO_0, 12), fontsize=10, color=AMBAR, va='center',
            arrowprops=dict(arrowstyle='->', color=AMBAR))
ax.set_xlim(0, W - 1); ax.set_ylim(0, 255); ax.set_xlabel('u [px]'); ax.set_ylabel('gris')
ax.set_title(f'El gris en la fila {FILA_P} (la línea blanca)', fontsize=12); ax.legend(loc='upper left', fontsize=10); ejes(ax)
plt.show()

**Qué tiene que salir:** las dos fotos y, con vuestro JSON, «Calibración de E⟨n⟩⟨letra⟩», sin ningún AVISO; en la figura, con la lámpara, la mesa se aclara de izquierda a derecha y la sombra queda a la izquierda del disco.

## 1. Una matriz de números, y el umbral

*En clase · diapositivas 23 y 24*

La foto en números; Otsu con el techo y con la lámpara, el umbral de antes y el adaptativo con una apertura.
Mirad cuántas piezas encuentra cada uno.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el umbral adaptativo: el lado del vecindario [px, impar] y lo que se suma a su media [niveles de gris]
VECINDARIO = 51  # @param {type:"number"}
MARGEN = 10  # @param {type:"number"}
# la apertura que borra las motas de la veta: el lado del cuadrado [px]
APERTURA = 3  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
MESA, CARA = int(np.median(I_t[V_t == 0])), int(np.median(I_t[V_t == 1]))   # el gris de la mesa y de la cara del disco
vd, ud = np.argwhere(V_t == 1).mean(0).round().astype(int)        # el centro del disco, en la verdad del simulador
v0, u0 = vd - 3, ud - 21                                          # 6 x 6 en su borde izquierdo (237 y 380 con la cámara del curso)
TROZO = I_t[v0:v0 + 6, u0:u0 + 6]

def encontradas(mask, V):
    """Cuántas piezas verdaderas cubre una componente de la máscara (IoU de las cajas de 0,5 o más)."""
    ps = piezas_en(mask, area_min=50, area_max=5000)
    def caja(k):
        ij = np.argwhere(V == k); return (ij[:, 1].min(), ij[:, 0].min(), ij[:, 1].max() + 1, ij[:, 0].max() + 1)
    def iou(A, B):
        ix = max(0, min(A[2], B[2]) - max(A[0], B[0])); iy = max(0, min(A[3], B[3]) - max(A[1], B[1]))
        return ix*iy/((A[2]-A[0])*(A[3]-A[1]) + (B[2]-B[0])*(B[3]-B[1]) - ix*iy)
    return sum(any(iou(p['caja'], caja(k)) >= 0.5 for p in ps) for k in range(1, V.max() + 1))

mal = lambda b, V: 100*((b > 0) != (V > 0)).mean()      # % de píxeles mal clasificados frente a la verdad
NUCLEO = np.ones((APERTURA, APERTURA), np.uint8)         # el de la apertura

th_t, b_t = cv2.threshold(I_t, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)       # Otsu, con la luz del techo
th_l, b_l = cv2.threshold(I_l, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)       # Otsu, con la lámpara
_, b_f = cv2.threshold(I_l, th_t, 255, cv2.THRESH_BINARY)                         # con la lámpara, el umbral de antes
b_a0 = cv2.adaptiveThreshold(I_l, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, VECINDARIO, -MARGEN)
b_a = cv2.morphologyEx(b_a0, cv2.MORPH_OPEN, NUCLEO)                              # el adaptativo, con la apertura
# de cada máscara: % mal clasificados y piezas encontradas de 3
MAL_T, PZ_T = mal(b_t, V_t), encontradas(b_t, V_t)
MAL_L, PZ_L = mal(b_l, V_l), encontradas(b_l, V_l)
MAL_F, PZ_F = mal(b_f, V_l), encontradas(b_f, V_l)
MAL_A, PZ_A = mal(b_a, V_l), encontradas(b_a, V_l)
MESA_L = (np.percentile(I_l[V_l == 0], 1), np.percentile(I_l[V_l == 0], 99))    # la mesa, con la lámpara: de y hasta
TROZOS_L = cv2.connectedComponents(b_l)[0] - 1                                   # los trozos de Otsu con la lámpara
CUADRADO_L = np.median(I_l[V_l == 2])                                            # el gris del cuadrado, con la lámpara
HIST_L = np.bincount(I_l.ravel(), minlength=256)                                 # el histograma, con la lámpara

# para las figuras: el histograma con el techo y cada máscara, píxel a píxel, frente a la verdad
from matplotlib.colors import to_rgb
HIST_T = np.bincount(I_t.ravel(), minlength=256)
RECORTE_6 = I_t[vd - 30:vd + 31, ud - 45:ud + 16].copy()                         # el disco, ampliado, con el trozo de 6 x 6 dentro
EXT_6 = (ud - 45.5, ud + 15.5, vd + 30.5, vd - 30.5)                              # su extent: el píxel (u, v) en (u, v)
COLORES_MAPA = np.array([[255*x for x in to_rgb(c)] for c in ('white', GRIS, ROJO, IQS_AZUL)]).round().astype(np.uint8)   # mesa, pieza, sobra, falta

def errores(b, V):
    """Para las figuras: blanco, la mesa; gris, la pieza; rojo, lo que sobra (mesa tomada por pieza); azul, lo que falta."""
    s, p = b > 0, V > 0
    return COLORES_MAPA[np.where(s, np.where(p, 1, 2), np.where(p, 3, 0))]

MAPAS = {'t': errores(b_t, V_t), 'l': errores(b_l, V_l), 'f': errores(b_f, V_l), 'a': errores(b_a, V_l)}

In [ ]:
# @title ✔ Resultado
print('La foto:', I_t.shape, I_t.dtype, '; la mesa ronda', MESA, 'y la cara del disco', CARA)
print(f'6 x 6 píxeles del borde izquierdo del disco (filas {v0} a {v0 + 5}, columnas {u0} a {u0 + 5}):')
print(TROZO)
print(f'\nLuz del techo: Otsu elige {th_t:.0f}; mal clasificados {coma(MAL_T)} %; piezas {PZ_T} de 3')
print(f'Con la lámpara, la mesa va de {MESA_L[0]:.0f} a {MESA_L[1]:.0f}:')
print(f'   Otsu elige {th_l:.0f}; mal {coma(MAL_L, 1)} %; {TROZOS_L} trozos; piezas {PZ_L} de 3')
print(f'   el umbral de antes ({th_t:.0f}): mal {coma(MAL_F)} %; piezas {PZ_F} de 3 (el cuadrado queda en {CUADRADO_L:.0f})')
print(f'   adaptativo ({VECINDARIO} x {VECINDARIO}, más {MARGEN}) y apertura de {APERTURA} x {APERTURA}: mal {coma(MAL_A)} %; piezas {PZ_A} de 3')

from matplotlib.patches import Patch, Rectangle
fig, axs = plt.subplots(1, 4, figsize=(15, 4.2), gridspec_kw=dict(width_ratios=[0.8, 1, 1.2, 1.2]))
ax = axs[0]                                       # dónde está el trozo, como en la diapositiva 23
ax.imshow(RECORTE_6, cmap='gray', vmin=0, vmax=255, extent=EXT_6)
ax.add_patch(Rectangle((u0 - 0.5, v0 - 0.5), 6, 6, fill=False, ec=ROJO, lw=2))
ax.set_xlabel('u [px]'); ax.set_ylabel('v [px]'); ax.set_title('El disco, ampliado (techo)', fontsize=12)
ax = axs[1]                                       # el trozo: la foto es una matriz de números
ax.imshow(TROZO, cmap='gray', vmin=0, vmax=255)
for i in range(6):
    for j in range(6):
        ax.text(j, i, TROZO[i, j], ha='center', va='center', fontsize=12, color='w' if TROZO[i, j] < 140 else 'k')
ax.set_xticks(range(6), [u0 + j for j in range(6)]); ax.set_yticks(range(6), [v0 + i for i in range(6)])
ax.set_xlabel('u [px]'); ax.set_ylabel('v [px]'); ax.set_title('6 × 6 píxeles de su borde (en rojo)', fontsize=12)
for sp in ax.spines.values():
    sp.set_color(ROJO); sp.set_linewidth(2)
# los histogramas, como en la diapositiva 24: Otsu en rojo; con la lámpara, el de antes en ámbar discontinuo
for ax, h_, col, t, th_ in zip(axs[2:], (HIST_T, HIST_L), (IQS_AZUL, AMBAR), ('con la luz del techo', 'con la lámpara'), (th_t, th_l)):
    ax.stairs(h_, np.arange(257) - 0.5, color=col, lw=1.3)
    ax.axvline(th_, color=ROJO, lw=1.8); ax.text(th_ + 3, 1.2e5, f'Otsu: {th_:.0f}', color=ROJO, fontsize=11, va='center')
    ax.set_yscale('log'); ax.set_ylim(0.8, 4e5); ax.set_yticks([1, 100, 10000], ['1', '100', '10 000'])
    ax.yaxis.set_minor_locator(plt.NullLocator()); ax.xaxis.set_major_formatter(COMA)
    ax.set_xlim(0, 255); ax.set_xlabel('gris'); ax.set_title(f'El histograma, {t}', fontsize=12)
axs[2].set_ylabel('píxeles')
axs[3].axvline(th_t, color=AMBAR, ls='--', lw=1.8)
axs[3].text(th_t + 3, 2.5e4, f'el de antes: {th_t:.0f}', color=AMBAR, fontsize=11, va='center')
plt.tight_layout(); plt.show()

fig, axs = plt.subplots(1, 4, figsize=(15, 3.7), gridspec_kw=dict(left=0.005, right=0.995, bottom=0.1, top=0.85, wspace=0.04))
for ax, (t, m, mal_, pz) in zip(axs, ((f'Techo, Otsu ({th_t:.0f})', MAPAS['t'], coma(MAL_T), PZ_T),
                                      (f'Lámpara, Otsu ({th_l:.0f})', MAPAS['l'], coma(MAL_L, 1), PZ_L),
                                      (f'Lámpara, el de antes ({th_t:.0f})', MAPAS['f'], coma(MAL_F), PZ_F),
                                      ('Lámpara, adaptativo', MAPAS['a'], coma(MAL_A), PZ_A))):
    ax.imshow(m); ax.set_title(f'{t}\n{mal_} % mal · piezas {pz} de 3', fontsize=11); sin_ejes(ax)
fig.legend(handles=[Patch(color=GRIS, label='pieza'), Patch(color=ROJO, label='sobra: mesa tomada por pieza'),
                    Patch(color=IQS_AZUL, label='falta: pieza tomada por mesa')], loc='lower center', ncol=3, fontsize=10, frameon=False)
plt.show()

**Qué tiene que salir:** con la cámara del curso, la mesa 89 y el disco 203; con el techo, Otsu 144 (3 de 3); con la lámpara, Otsu 68 (45,5 % mal, 1 de 3), el 144 (2 de 3) y el adaptativo (3 de 3); en la figura, con 68 sobra media mesa y con 144 falta el cuadrado.

## 2. De la máscara a la pieza, y de la pieza al robot

*En clase, con el ejercicio de clase · diapositivas 25 y 26*

De la máscara al disco; de su centroide a la mesa, cortando con la cara, y al 2R.

### Componentes, momentos y circularidad

La apertura borra las motas; la regla del disco lo separa de las otras piezas.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# los márgenes del umbral adaptativo que se comparan, antes y después de la apertura [niveles de gris]
MARGENES = [8, 10]  # @param {type:"raw"}
# la regla del disco: circularidad mínima y área de la cara, de y hasta [px]
CIRC_MIN = 0.85  # @param {type:"number"}
AREA_DISCO = (500, 1200)  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
MOTAS = []                                       # (margen, componentes antes y después de la apertura)
for margen in MARGENES:
    b0 = cv2.adaptiveThreshold(I_l, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, VECINDARIO, -margen)
    n0 = cv2.connectedComponents(b0)[0] - 1
    n1 = cv2.connectedComponents(cv2.morphologyEx(b0, cv2.MORPH_OPEN, NUCLEO))[0] - 1
    MOTAS.append((margen, n0, n1))
# para la figura, con el último margen: las etiquetas antes y después de la apertura, engrosadas 5 x 5 para ver las motas
VER_MOTAS = [cv2.dilate(cv2.connectedComponents(m_)[1].astype(np.float32), np.ones((5, 5), np.uint8)).astype(int)
             for m_ in (b0, cv2.morphologyEx(b0, cv2.MORPH_OPEN, NUCLEO))]

nombres = {1: 'disco', 2: 'cuadrado', 3: 'triángulo'}
PIEZAS_T = [(nombres.get(V_t[int(round(p['v'])), int(round(p['u']))], '?'), p) for p in piezas_en(b_t)]   # techo y Otsu

def el_disco(b):
    """La componente que pasa la regla del disco: circularidad mayor que CIRC_MIN y área entre los dos de AREA_DISCO."""
    d = [p for p in piezas_en(b) if p['circ'] > CIRC_MIN and AREA_DISCO[0] < p['area'] < AREA_DISCO[1]]
    return d[0] if d else None

disco = el_disco(b_t)                            # el disco, con la luz del techo y Otsu
assert disco is not None, 'Ninguna componente pasa la regla del disco: bajad CIRC_MIN o ampliad AREA_DISCO'
# los momentos, a mano: sumas sobre los píxeles del disco
_, et = cv2.connectedComponents(b_t)
k = et[int(round(disco['v'])), int(round(disco['u']))]
v, u = np.nonzero(et == k)
m00, m10, m01 = len(u), u.sum(), v.sum()
U_MANO, V_MANO = m10/m00, m01/m00                # el centroide, a mano [px]
uc, vc = disco['u'], disco['v']                  # el centroide, con cv2.moments [px]
# para la figura: el contorno de cada pieza (el del perímetro) y si pasa la regla del disco
CONT_T = [cv2.findContours((et == et[int(round(p_['v'])), int(round(p_['u']))]).astype(np.uint8), cv2.RETR_EXTERNAL,
                           cv2.CHAIN_APPROX_NONE)[0][0][:, 0] for _, p_ in PIEZAS_T]
PASA_T = [p_['circ'] > CIRC_MIN and AREA_DISCO[0] < p_['area'] < AREA_DISCO[1] for _, p_ in PIEZAS_T]

In [ ]:
# @title ✔ Resultado
for margen, n0, n1 in MOTAS:
    print(f'adaptativo más {margen}: {n0} componentes; tras la apertura de {APERTURA} x {APERTURA}, {n1}')
print('\nCon la luz del techo y Otsu, pieza a pieza:')
for nom, p in PIEZAS_T:
    print(f"  {nom:10s} área m00 = {p['area']:5.0f} px ; centroide ({coma(p['u'])}; {coma(p['v'])}) ; circularidad {coma(p['circ'], 3)}")
print(f'\nA mano: m00 = {m00}, m10/m00 = {coma(U_MANO)}, m01/m00 = {coma(V_MANO)}  (cv2.moments da lo mismo)')

PAL_ET = (255*np.vstack([[0, 0, 0], plt.cm.tab20(np.arange(20))[:, :3]])).round().astype(np.uint8)   # un color por etiqueta; en negro, la mesa
margen, n0, n1 = MOTAS[-1]                        # la figura, con el último margen de la lista
fig, axs = plt.subplots(1, 2, figsize=(11, 4.4), gridspec_kw=dict(left=0.01, right=0.99, bottom=0.01, top=0.86, wspace=0.04))
for ax, et_, t in zip(axs, VER_MOTAS, (f'más {margen}: {n0} componentes', f'más {margen}, tras la apertura de {APERTURA} × {APERTURA}: {n1}')):
    ax.imshow(PAL_ET[np.where(et_ > 0, 1 + (et_ - 1) % 20, 0)]); ax.set_title(t, fontsize=12); sin_ejes(ax)
fig.suptitle('El adaptativo con la lámpara: un color por componente (engrosadas 5 × 5 para ver las motas)', fontsize=12)
plt.show()

cajas_ = np.array([p['caja'] for _, p in PIEZAS_T])
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.imshow(FOTOS_0[0], cmap='gray', vmin=0, vmax=255)
for (nom, p), c_, pasa in zip(PIEZAS_T, CONT_T, PASA_T):
    col = IQS_VERDE if pasa else ROJO
    ax.plot(*np.vstack([c_, c_[:1]]).T, '-', color=col, lw=1.5)
    ax.plot(p['u'], p['v'], '+', color='k', ms=12, mew=2)
    ax.text(p['caja'][2] + 6, p['v'], f"{nom}\n{p['area']:.0f} px\ncirc. {coma(p['circ'])}", fontsize=11, color=col, va='center',
            bbox=dict(fc='w', ec='none', alpha=0.85, pad=1.5))
ax.set_xlim(max(cajas_[:, 0].min() - 30, -0.5), min(cajas_[:, 2].max() + 100, W - 0.5))
ax.set_ylim(min(cajas_[:, 3].max() + 25, H - 0.5), max(cajas_[:, 1].min() - 25, -0.5))
ax.set_xlabel('u [px]'); ax.set_ylabel('v [px]')
ax.set_title(f'Techo y Otsu: contorno, centroide (+) y circularidad; en verde, lo que pasa la regla del disco\n'
             f'(circularidad mayor que {coma(CIRC_MIN)} y área de {AREA_DISCO[0]} a {AREA_DISCO[1]} px); en rojo, lo que no', fontsize=11)
plt.show()

**Qué tiene que salir:** con la cámara del curso, con más 10, de 97 componentes quedan 3; circularidades 0,93, 0,73 y 0,61; el disco, 813 px, en (400,59; 240,01); en la figura, las motas se borran con la apertura y solo el disco pasa la regla (en verde).

### Del centroide a la mesa y al robot

Con la calibración y la inversa de S10; y cuánto tiembla el centroide de foto a foto.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# fotos iguales, con distinto ruido del sensor, para medir cuánto tiembla el centroide
N_FOTOS = 30  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
# CAL = (K, distorsión, T_C0): la de la cámara del curso o la de uno de los dos en S16
Pm = a_la_mesa((uc, vc), *CAL, z=H_PIEZA)[0]                      # cortando el rayo con la cara, a 10 mm [m]
q1, q2 = ik2r(*Pm, +1)                                            # la inversa de S10, codo abajo [rad]
Q_GRADOS = np.degrees([q1, q2])
Psd = a_la_mesa((uc, vc), CAL[0], np.zeros(5), CAL[2], z=H_PIEZA)[0]   # sin quitar la distorsión (solo con la K)
ERR_P, ERR_SD = 1e3*np.linalg.norm(Pm - P_PIEZA), 1e3*np.linalg.norm(Psd - P_PIEZA)    # a cuánto de P [mm]
cs = []                                                           # el centroide en cada foto
for s in range(int(N_FOTOS)):
    I_ = cam.foto(rng=np.random.default_rng(1000 + s))
    _, b_ = cv2.threshold(I_, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    d_ = [p for p in piezas_en(b_) if p['circ'] > CIRC_MIN][0]
    cs.append((d_['u'], d_['v']))
TIEMBLA = np.std(cs, 0)                                           # σ del centroide en u y en v [px]
CODO = A1*np.array([np.cos(q1), np.sin(q1)])                      # para la figura: dónde queda el codo [m]
NUBE = np.array(cs)                                               # y los centroides de las fotos [px]

In [ ]:
# @title ✔ Resultado
print(f'El disco, en la mesa: ({coma(Pm[0], 4)}; {coma(Pm[1], 4)}) m, a {coma(ERR_P)} mm de P')
print(f'q (codo abajo) = ({coma(Q_GRADOS[0])}°; {coma(Q_GRADOS[1])}°)')
print(f'Sin quitar la distorsión: a {coma(ERR_SD)} mm de P')
print(f'El centroide tiembla σ = ({coma(TIEMBLA[0], 3)}; {coma(TIEMBLA[1], 3)}) px de foto a foto')

from matplotlib.patches import Circle, Ellipse
fig, axs = plt.subplots(1, 3, figsize=(15, 4.7), gridspec_kw=dict(width_ratios=[1.45, 1, 1]))
ax = axs[0]                                       # el 2R de S10, en la mesa, vista desde arriba
ax.add_patch(Circle(P_PIEZA, D_PIEZA/2, fc='none', ec=IQS_VERDE, lw=1.5, label='la pieza, en P'))
ax.plot([0, CODO[0], Pm[0]], [0, CODO[1], Pm[1]], '-o', color=IQS_AZUL, lw=4, ms=8, mfc='w', mew=2, label='el 2R, hasta lo medido')
ax.plot(0, 0, 's', color='k', ms=11)
ax.text(0.04, -0.035, f'q1 = {coma(Q_GRADOS[0])}°', fontsize=11, color=IQS_AZUL)
ax.text(CODO[0] + 0.025, CODO[1] - 0.03, f'q2 = {coma(Q_GRADOS[1])}°', fontsize=11, color=IQS_AZUL)
ax.set_aspect('equal'); ax.set_xlim(-0.05, 0.62); ax.set_ylim(-0.08, 0.3); ejes(ax)
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]'); ax.legend(loc='upper left', fontsize=10)
ax.set_title('La mesa desde arriba: el robot llega a la pieza', fontsize=12)
ax = axs[1]                                       # alrededor de P, en milímetros
L_ = max(0.3, 1.4*max(ERR_P, ERR_SD))
ax.plot(*1e3*P_PIEZA, '+', color=IQS_VERDE, ms=20, mew=2.5, label='P')
ax.plot(*1e3*Pm, 'o', color=IQS_AZUL, ms=8, label=f'medido: a {coma(ERR_P)} mm')
ax.plot(*1e3*Psd, 'x', color=ROJO, ms=10, mew=2.5, label=f'sin quitar la distorsión: a {coma(ERR_SD)} mm')
ax.set_xlim(1e3*P_PIEZA[0] - L_, 1e3*P_PIEZA[0] + L_); ax.set_ylim(1e3*P_PIEZA[1] - L_, 1e3*P_PIEZA[1] + L_)
ax.set_aspect('equal'); ax.grid(True, color='#E6E6E6'); ejes(ax); ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), fontsize=10, frameon=False)
ax.set_title('Alrededor de P, ampliado', fontsize=12)
ax = axs[2]                                       # el centroide de cada foto, en píxeles
mu_ = NUBE.mean(0); R_ = max(4*TIEMBLA.max(), 0.05)
ax.plot(NUBE[:, 0], NUBE[:, 1], 'o', color=IQS_AZUL, ms=4, alpha=0.75)
ax.plot(*mu_, '+', color='k', ms=14, mew=2)
ax.add_patch(Ellipse(mu_, 2*TIEMBLA[0], 2*TIEMBLA[1], fc='none', ec='k', ls='--', lw=1.3))
ax.set_xlim(mu_[0] - R_, mu_[0] + R_); ax.set_ylim(mu_[1] + R_, mu_[1] - R_); ax.set_aspect('equal')
ax.grid(True, color='#E6E6E6'); ejes(ax); ax.set_xlabel('u [px]'); ax.set_ylabel('v [px]'); ax.locator_params(nbins=5)
ax.set_title(f'El centroide en {len(NUBE)} fotos, ampliado: la media (+)\ny la elipse de σ = ({coma(TIEMBLA[0], 3)}; {coma(TIEMBLA[1], 3)}) px', fontsize=12)
plt.tight_layout(); plt.show()

**Qué tiene que salir:** con la cámara del curso, a 0,02 mm de P y q = (1,04°; 74,47°); sin quitar la distorsión, 0,27 mm; el temblor, 0,03 px; en la figura, lo medido casi encima de P y la nube de centroides, de unas centésimas de píxel.

### Ejercicio de clase

*En clase · diapositiva 32; en común a la vuelta*

Con la lámpara, ¿a cuántos milímetros de P sale el disco con Otsu, el 144 de antes y el adaptativo, frente al temblor del centroide?
Con vuestro JSON: poned `EQUIPO` y `LETRA` en ▶ Parámetros (sección 0), volved a ejecutar desde allí y mirad la última línea.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
EJ = []                                          # (método, el disco o None, a cuánto de P [mm])
for nom, b in ((f'Otsu ({th_l:.0f})', b_l), (f'el umbral de antes ({th_t:.0f})', b_f), ('el adaptativo', b_a)):
    d = el_disco(b)
    if d is None:                                # ninguna componente pasa la regla del disco
        EJ.append((nom, None, None))
        continue
    Pq = a_la_mesa((d['u'], d['v']), *CAL, z=H_PIEZA)[0]
    EJ.append((nom, d, 1e3*np.linalg.norm(Pq - P_PIEZA)))
Psc = a_la_mesa((uc, vc), K_CURSO, np.zeros(5), T_C0_verdad(), z=H_PIEZA)[0]   # sin calibrar: hoja de datos y plano del montaje
ERR_SC = 1e3*np.linalg.norm(Psc - P_PIEZA)       # [mm]
DEC_SC = 2 if np.linalg.norm(Psc - P_PIEZA) < 1e-3 else 1
# para la figura: el disco ampliado con cada máscara frente a la verdad, y el temblor del centroide en milímetros
EJ_ZONA = (slice(vd - 24, vd + 25), slice(ud - 24, ud + 25))
EJ_MAPAS = [MAPAS[c][EJ_ZONA] for c in 'lfa']    # Otsu, el de antes y el adaptativo, como en EJ
EJ_VERDAD = []                                   # el contorno de la cara del disco: los lados de píxel entre la cara y lo demás, (u, v)
m_, v0_, u0_ = VERDAD_0[1][EJ_ZONA] == 1, EJ_ZONA[0].start, EJ_ZONA[1].start   # la de la sección 0: la 3 cambia V_l
for i_, j_ in zip(*np.nonzero(m_[1:, :] != m_[:-1, :])):          # entre las filas i_ e i_ + 1
    EJ_VERDAD.append([(u0_ + j_ - 0.5, v0_ + i_ + 0.5), (u0_ + j_ + 0.5, v0_ + i_ + 0.5)])
for i_, j_ in zip(*np.nonzero(m_[:, 1:] != m_[:, :-1])):          # entre las columnas j_ y j_ + 1
    EJ_VERDAD.append([(u0_ + j_ + 0.5, v0_ + i_ - 0.5), (u0_ + j_ + 0.5, v0_ + i_ + 0.5)])
MM_POR_PX = 1e3*np.linalg.norm(np.diff(a_la_mesa([(disco['u'], disco['v']), (disco['u'] + 1, disco['v'])], *CAL, z=H_PIEZA), axis=0))   # junto a la pieza [mm/px]
TEMBLOR_MM = MM_POR_PX*TIEMBLA.max()             # [mm]

In [ ]:
# @title ✔ Resultado
for nom, d, mm in EJ:
    if d is None:
        print(f'{nom}: ninguna componente pasa la regla: el disco no aparece')
    else:
        print(f"{nom}: centroide ({coma(d['u'])}; {coma(d['v'])}), {d['area']:.0f} px ; la pieza a {coma(mm)} mm de P")
print(f'Con la luz del techo: con {CON}, a {coma(ERR_P)} mm ; sin calibrar, a {coma(ERR_SC, DEC_SC)} mm')

from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.collections import LineCollection
fig, axs = plt.subplots(1, 5, figsize=(15, 4.6), gridspec_kw=dict(width_ratios=[1, 1, 1, 0.8, 1.6], left=0.045, right=0.99,
                                                                  bottom=0.2, top=0.85, wspace=0.1))
axs[3].axis('off')                                # el sitio de los rótulos de las barras
ext_ = (EJ_ZONA[1].start - 0.5, EJ_ZONA[1].stop - 0.5, EJ_ZONA[0].stop - 0.5, EJ_ZONA[0].start - 0.5)
for ax, (nom, d, mm), m in zip(axs, EJ, EJ_MAPAS):
    ax.imshow(m, extent=ext_)
    ax.add_collection(LineCollection(EJ_VERDAD, colors=IQS_VERDE, linewidths=1.5))
    if d is None:
        ax.text(0.5, 0.06, 'no aparece', transform=ax.transAxes, ha='center', va='bottom', fontsize=12,
                bbox=dict(fc='w', ec='k', pad=3))
    else:
        ax.plot(d['u'], d['v'], '+', color='k', ms=16, mew=2)
    ax.set_title(nom[0].upper() + nom[1:], fontsize=12); ax.set_xlabel('u [px]')
axs[0].set_ylabel('v [px]'); axs[1].tick_params(labelleft=False); axs[2].tick_params(labelleft=False)
ax = axs[4]                                       # a cuántos milímetros de P
barras_ = [(nom, mm, 2, AMBAR) for nom, d, mm in EJ] + [(f'techo, con {CON}', ERR_P, 2, '#555555'), ('techo, sin calibrar', ERR_SC, DEC_SC, '#555555')]
val_ = [v_ for v_ in [mm for _, mm, _, _ in barras_] + [TEMBLOR_MM] if v_]          # lo que se dibuja en escala logarítmica
lo_, hi_ = min(val_)/3, max(val_)*8
ys_ = np.arange(len(barras_))[::-1]
for y_, (nom, mm, n_, col) in zip(ys_, barras_):
    if mm is None:
        ax.text(lo_*1.15, y_, 'no aparece', va='center', fontsize=10, color=GRIS, zorder=3, bbox=dict(fc='w', ec='none', pad=1))
    else:
        ax.barh(y_, mm - lo_, left=lo_, height=0.6, color=col)
        ax.text(mm*1.12, y_, f'{coma(mm, n_)} mm', va='center', fontsize=10, bbox=dict(fc='w', ec='none', pad=0.5, alpha=0.8))
if TEMBLOR_MM > 0:
    ax.axvline(TEMBLOR_MM, color='k', ls='--', lw=1.5)
    ax.text(TEMBLOR_MM*1.1, -0.95, f'el temblor del centroide:\nσ ≈ {coma(TEMBLOR_MM, 2)} mm', fontsize=10, va='center')
ax.set_xscale('log'); ax.set_xlim(lo_, hi_); ax.set_ylim(-1.5, len(barras_) - 0.4)
ax.xaxis.set_major_formatter(COMA); ax.xaxis.set_minor_locator(plt.NullLocator())
ax.set_yticks(ys_, [nom.replace(' (', '\n(') for nom, _, _, _ in barras_], fontsize=10)
ax.set_xlabel('distancia a P [mm], escala logarítmica'); ax.set_title('La pieza, a cuántos mm de P', fontsize=12)
fig.legend(handles=[Patch(color=GRIS, label='pieza'), Patch(color=ROJO, label='sobra'), Patch(color=IQS_AZUL, label='falta'),
                    Line2D([], [], color=IQS_VERDE, lw=2, label='la cara del disco, de verdad'),
                    Line2D([], [], color='k', marker='+', ms=12, mew=2, ls='none', label='el centroide')],
           loc='lower center', ncol=5, fontsize=10, frameon=False)
fig.suptitle('El disco, ampliado, con cada máscara de la lámpara; y a cuántos milímetros de P sale la pieza', fontsize=12)
plt.show()

### ✎ Vuestra respuesta (dos líneas)

*Doble clic aquí y escribid, con cifras. Se sube al acabar la clase, con el S17b.*

1. Con la lámpara: a cuántos milímetros de P sale el disco con Otsu, con el 144 y con el adaptativo, y por qué.
2. Con vuestra calibración de S16 (o la del curso) y sin calibrar: a cuántos milímetros, y qué pesa más.

## 3. La convolución, los bordes y Canny

*En clase · diapositivas 27 a 30; con la cámara del curso, sea cual sea `EQUIPO`*

Una convolución a mano, el suavizado, el gradiente y Canny. Mirad cuánta veta pasa por borde.

### La convolución y el suavizado

Sobel en el borde del disco; y cada filtro, con el ruido de la mesa y la anchura del borde.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
if EQUIPO != 0:      # las secciones 3 y 4 son las cuentas de las diapositivas 27 a 31: siempre con la foto de la cámara del curso
    cam0 = Mesa()
    I_t, V_t = cam0.foto(mascara=True)
    I_l, V_l = cam0.foto(rng=np.random.default_rng(1), mascara=True, **LAMPARA)
    _, b_t = cv2.threshold(I_t, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    uc, vc = [(p['u'], p['v']) for p in piezas_en(b_t) if p['circ'] > 0.85][0]
SOBEL = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
ub, vb = int(round(uc)) - 17, int(round(vc)) - 1          # el borde izquierdo del disco (384; 239 con la cámara del curso)
P3 = I_t[vb-1:vb+2, ub-1:ub+2].astype(int)
SOBEL_MANO = (SOBEL*P3).sum()                             # la suma, a mano
SOBEL_8 = SOBEL_MANO/8                                    # en niveles por píxel
FILTRO = cv2.filter2D(I_t.astype(float), -1, SOBEL)[vb, ub]   # lo mismo, con OpenCV
SUMAS = (np.full((3, 3), 1/9).sum(), (np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]])/16).sum(), SOBEL.sum())   # caja, gaussiana, Sobel
SOBEL_P3 = SOBEL*P3                                       # para la figura: los nueve productos de la suma

def anchura_borde(J, v=240, u0=376, u1=392):
    """Del 10 al 90 % del salto del borde izquierdo del disco, en píxeles."""
    pr = J[v, u0:u1].astype(float); lo, hi = pr[:3].mean(), pr[-3:].mean()
    x = np.arange(len(pr))
    return np.interp(lo + 0.9*(hi - lo), pr, x) - np.interp(lo + 0.1*(hi - lo), pr, x)

zona = (slice(330, 400), slice(200, 300))                     # un trozo de mesa sin piezas
SUAVIZADOS = []                                               # (filtro, σ de la mesa [niveles], anchura del borde [px])
SUAV_VER = []                                                 # para la figura: (la zona de mesa, la fila 240 de u = 360 a 399)
for nom, J in (('sin suavizar', I_t.astype(float)), ('gaussiana σ = 1', cv2.GaussianBlur(I_t.astype(float), (0, 0), 1)),
               ('gaussiana σ = 2', cv2.GaussianBlur(I_t.astype(float), (0, 0), 2)), ('gaussiana σ = 4', cv2.GaussianBlur(I_t.astype(float), (0, 0), 4)),
               ('caja de 7 x 7', cv2.blur(I_t.astype(float), (7, 7)))):
    SUAVIZADOS.append((nom, J[zona].std(), anchura_borde(J)))
    SUAV_VER.append((J[zona].copy(), J[240, 360:400].copy()))

In [ ]:
# @title ✔ Resultado
if EQUIPO != 0:
    print('Secciones 3 y 4: con la foto de la cámara del curso, la de las diapositivas (vuestra unidad, en las secciones 1 y 2).')
print(f'3 x 3 píxeles del borde del disco, alrededor de ({ub}; {vb}):\n', P3)
print('con Sobel en u, a mano:', int(SOBEL_MANO), '; entre 8:', coma(SOBEL_8, 1), 'niveles por píxel')
print('cv2.filter2D en ese píxel:', coma(FILTRO, 0))
print('sumas: caja', coma(SUMAS[0], 0), '; gaussiana', coma(SUMAS[1], 0), '; Sobel', coma(SUMAS[2], 0))
print('\nSuavizar: el gris de la mesa (ruido y veta) y la anchura del borde')
for nom, s_mesa, borde in SUAVIZADOS:
    print(f'  {nom:16s} σ de la mesa {coma(s_mesa)} niveles ; borde de {coma(borde, 1)} px')

fig, axs = plt.subplots(1, 3, figsize=(11, 3.7), gridspec_kw=dict(left=0.01, right=0.99, bottom=0.01, top=0.8, wspace=0.08))
for ax, (M_, t, cm, lim) in zip(axs, ((P3, f'La foto: 3 × 3 en ({ub}; {vb})', 'gray', 255),
                                      (SOBEL, '× el núcleo: Sobel en u', 'RdBu_r', 2.6),
                                      (SOBEL_P3, f'= los productos: suman {int(SOBEL_MANO)}', 'RdBu_r', 400))):
    ax.imshow(M_, cmap=cm, vmin=0 if cm == 'gray' else -lim, vmax=lim)
    for i in range(3):
        for j in range(3):
            claro = M_[i, j] < 140 if cm == 'gray' else abs(M_[i, j]) > 0.6*lim
            ax.text(j, i, f'{M_[i, j]:g}'.replace('-', '−'), ha='center', va='center', fontsize=14, color='w' if claro else 'k')
    ax.set_title(t, fontsize=12); sin_ejes(ax)
fig.suptitle(f'Una convolución en un píxel del borde: {int(SOBEL_MANO)}; entre 8, {coma(SOBEL_8, 1)} niveles por píxel', fontsize=12)
plt.show()

fig = plt.figure(figsize=(15, 6.4))
gs = fig.add_gridspec(2, len(SUAVIZADOS), height_ratios=[1, 1.1], left=0.05, right=0.99, bottom=0.08, top=0.85, wspace=0.08, hspace=0.45)
lo_, hi_ = SUAV_VER[0][0].min(), SUAV_VER[0][0].max()     # la misma escala de gris en los cinco
for k_, ((nom, s_mesa, borde), (rec, _)) in enumerate(zip(SUAVIZADOS, SUAV_VER)):
    ax = fig.add_subplot(gs[0, k_])
    ax.imshow(rec, cmap='gray', vmin=lo_, vmax=hi_, extent=(zona[1].start - 0.5, zona[1].stop - 0.5, zona[0].stop - 0.5, zona[0].start - 0.5))
    ax.set_title(nom.replace(' x ', ' × ') + f'\nruido de la mesa: {coma(s_mesa)} niveles', fontsize=11)
    if k_:
        sin_ejes(ax)
    else:
        ax.set_xlabel('u [px]'); ax.set_ylabel('v [px]')
ax = fig.add_subplot(gs[1, :])
for (nom, s_mesa, borde), (_, pr), col in zip(SUAVIZADOS, SUAV_VER, (GRIS, IQS_VERDE, IQS_AZUL, AMBAR, 'k')):
    ax.plot(np.arange(360, 400), pr, color=col, lw=2, ls='--' if 'caja' in nom else '-', label=nom.replace(' x ', ' × ') + f': borde de {coma(borde, 1)} px')
ax.set_xlim(360, 399); ax.set_xlabel('u [px], en la fila 240'); ax.set_ylabel('gris'); ejes(ax)
ax.legend(fontsize=10, ncol=2, loc='upper left', frameon=False)
ax.set_title('El borde izquierdo del disco, en la fila 240', fontsize=12)
fig.suptitle(f'Arriba, la misma zona de mesa (sin piezas), con la misma escala de gris ({lo_:.0f} a {hi_:.0f})', fontsize=12)
plt.show()

**Qué tiene que salir:** Sobel en u, 365 (45,6 niveles por píxel); la mesa, de 4,33 niveles a 0,15 con σ = 4, y el borde, de 2,0 a 8,4 px; en la figura, la veta se borra y el borde se tumba al crecer σ.

### El gradiente y la sombra

Con la lámpara, la fila 240 cruza la sombra y el disco.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
def gradiente(I, s=1.0):
    J = cv2.GaussianBlur(I.astype(float), (0, 0), s) if s > 0 else I.astype(float)
    Iu, Iv = cv2.Sobel(J, cv2.CV_64F, 1, 0)/8, cv2.Sobel(J, cv2.CV_64F, 0, 1)/8
    return Iu, Iv, np.hypot(Iu, Iv)

_, _, G_l = gradiente(I_l)                       # ‖∇I‖ con la lámpara, con σ = 1 [niveles por píxel]
FILA_GRIS = I_l[240, 365:390]                    # la fila 240, de u = 365 a 389
FILA_GRAD = np.round(G_l[240, 365:390]).astype(int)
G_SOMBRA, G_DISCO, G_MESA = G_l[240, 365:378].max(), G_l[240, 378:390].max(), G_l[zona].max()
# para la figura: la fila 240 sin redondear, y dónde están sus dos picos
FILA_GRAD_F = G_l[240, 365:390].copy()
U_SOMBRA, U_DISCO = 365 + int(np.argmax(G_l[240, 365:378])), 378 + int(np.argmax(G_l[240, 378:390]))

In [ ]:
# @title ✔ Resultado
print('Con la lámpara, fila 240: el gris y ‖∇I‖ de u = 365 a 389')
print(' ', FILA_GRIS)
print(' ', FILA_GRAD)
print('  la sombra da un borde de', coma(G_SOMBRA, 1), 'niveles por píxel; el disco,', coma(G_DISCO, 1),
      '; la mesa, como mucho', coma(G_MESA, 1))

fig, (ax0, ax) = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw=dict(width_ratios=[1, 1.45], wspace=0.3))
im_ = ax0.imshow(G_l[200:281, 330:451], cmap='magma', vmin=0, vmax=40, extent=(329.5, 450.5, 280.5, 199.5))   # como en la diapositiva 29
ax0.plot([365, 389], [240, 240], color='w', ls='--', lw=1.5)
ax0.set_xlabel('u [px]'); ax0.set_ylabel('v [px]'); ax0.set_title('‖∇I‖ con la lámpara, ampliado [px];\nla línea blanca, la fila 240 de la derecha', fontsize=12)
fig.colorbar(im_, ax=ax0, shrink=0.9, pad=0.03).set_label('‖∇I‖ [niveles/px]')
uu_ = np.arange(365, 390)
ax.plot(uu_, FILA_GRIS, '-o', color=GRIS, lw=2, ms=3)
ax2 = ax.twinx(); ax2.plot(uu_, FILA_GRAD_F, '-o', color=ROJO, lw=2, ms=3)
ax2.axhline(G_MESA, color=ROJO, ls=':', lw=1.2)
ax2.text(373.2, G_MESA + 0.6, f'la veta, como mucho {coma(G_MESA, 1)}', color=ROJO, fontsize=10, va='bottom')
ax2.annotate(f'la sombra: {coma(G_SOMBRA, 1)}', (U_SOMBRA, G_SOMBRA), (U_SOMBRA + 2.5, G_SOMBRA + 13), color=ROJO, fontsize=11,
             arrowprops=dict(arrowstyle='->', color=ROJO))
ax2.annotate(f'el disco: {coma(G_DISCO, 1)}', (U_DISCO, G_DISCO), (U_DISCO - 1.5, G_DISCO + 4), color=ROJO, fontsize=11, ha='right',
             arrowprops=dict(arrowstyle='->', color=ROJO))
ax.set_ylim(0, 200); ax2.set_ylim(0, 45); ax.set_xlim(364.5, 389.5)
ax.set_xlabel('u [px], en la fila 240'); ax.set_ylabel('gris', color=GRIS); ax2.set_ylabel('‖∇I‖ con σ = 1 [niveles/px]', color=ROJO)
ejes(ax, ax2); ax.set_title('Con la lámpara, la fila 240 cruza la sombra y el disco:\nel gris y ‖∇I‖', fontsize=12)
plt.show()

**Qué tiene que salir:** la sombra, 11,0 niveles por píxel; el disco, 36,6; la veta, como mucho 2,8; en la figura, dos anillos, el de la sombra y el del disco, y en la fila, un pico en la sombra y otro, más alto, en el disco.

### Canny

Cambiad σ y los umbrales; mirad los bordes en la mesa, lejos de las piezas.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# las anchuras σ del suavizado que se prueban [px]; 0, sin suavizar
SIGMAS = [0, 1, 2, 4]  # @param {type:"raw"}
# los dos umbrales de la histéresis [niveles por píxel]; cv2.Canny los quiere por 8
BAJO = 3  # @param {type:"number"}
ALTO = 8  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
lejos = cv2.dilate((V_l > 0).astype(np.uint8), np.ones((31, 31), np.uint8)) == 0   # la mesa, lejos de las piezas
cannys, CANNY_PX = {}, {}                        # σ -> los bordes; y (píxeles de borde, de ellos en la mesa lejos)
for s in SIGMAS:
    J = cv2.GaussianBlur(I_l, (0, 0), s) if s else I_l
    cannys[s] = cv2.Canny(J, BAJO*8, ALTO*8, L2gradient=True)
    CANNY_PX[s] = (int((cannys[s] > 0).sum()), int((cannys[s][lejos] > 0).sum()))
_, _, G1 = gradiente(I_l, 1)
BANDAS = (G1 > BAJO)*255                         # ‖∇I‖ por encima del umbral bajo, sin Canny
N_BANDAS = int((G1 > BAJO).sum())
MUESTRA = list(dict.fromkeys(list(SIGMAS[:2]) + list(SIGMAS[-1:])))   # los σ de la figura: los dos primeros y el último
# para la figura: los bordes de esos σ, recortados y engrosados a 2 px para que se vean (las cuentas, con los de 1 px)
VER_CANNY = {s: cv2.dilate(cannys[s][60:480, 100:640], np.ones((2, 2), np.uint8)) for s in MUESTRA}

In [ ]:
# @title ✔ Resultado
for s in SIGMAS:
    print(f'Canny con σ = {s}: {CANNY_PX[s][0]} px de borde; {CANNY_PX[s][1]} en la mesa, lejos de las piezas')
print(f'‖∇I‖ > {BAJO:g} con σ = 1, sin Canny:', N_BANDAS, 'px, en bandas anchas')

fig, axs = plt.subplots(1, 1 + len(MUESTRA), figsize=(15, 3.9))
for ax, (t, b) in zip(axs, [(f'‖∇I‖ > {BAJO:g}, σ = 1: {N_BANDAS} px\n(sin Canny)', BANDAS[60:480, 100:640])] +
                           [(f'Canny, σ = {s}: {CANNY_PX[s][0]} px\n({CANNY_PX[s][1]} en la mesa)', VER_CANNY[s]) for s in MUESTRA]):
    ax.imshow(255 - b, cmap='gray', vmin=0, vmax=255); ax.set_title(t, fontsize=12); sin_ejes(ax)
fig.suptitle('Los bordes, con la lámpara (recorte: u de 100 a 639, v de 60 a 479; los de Canny, engrosados a 2 px)', fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.92)); plt.show()

**Qué tiene que salir:** con σ = 0, 1, 2 y 4, 791, 560, 415 y 263 px de borde; en la veta, 207 con σ = 0 y ninguno con σ ≥ 1; en la figura, la veta de abajo a la derecha solo con σ = 0.

## 4. Las esquinas: Harris (en casa)

*En casa · diapositiva 31*

La matriz de estructura en la mesa, en un borde y en una esquina; Harris, girando la foto y a la mitad de tamaño.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# cuántas esquinas se buscan, como mucho
N_ESQUINAS = 50  # @param {type:"number"}
# el giro de la foto [°]
GIRO = 30  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
Js = cv2.GaussianBlur(I_t, (0, 0), 1).astype(np.float32)
Ix, Iy = cv2.Sobel(Js, cv2.CV_64F, 1, 0)/8, cv2.Sobel(Js, cv2.CV_64F, 0, 1)/8

def M_est(u, v, w=3):
    s = (slice(v - w, v + w + 1), slice(u - w, u + w + 1)); a, b = Ix[s], Iy[s]
    return np.array([[(a*a).sum(), (a*b).sum()], [(a*b).sum(), (b*b).sum()]])

SITIOS = []                                      # (sitio, autovalores de M, c de Harris)
SITIOS_UV = []                                   # para la figura: dónde está cada sitio (u, v) [px]
for nom, (u, v) in (('la mesa', (250, 380)), ('el borde del disco', (385, 240)), ('la esquina del cuadrado', (157, 119))):
    M = M_est(u, v); lam = np.linalg.eigvalsh(M); c = np.linalg.det(M) - 0.04*np.trace(M)**2
    SITIOS.append((nom, lam, c))
    SITIOS_UV.append((u, v))

def esquinas(I, n):
    q = cv2.goodFeaturesToTrack(I, maxCorners=int(n), qualityLevel=0.05, minDistance=8, blockSize=7, useHarrisDetector=True, k=0.04)
    return np.zeros((0, 2)) if q is None else q.reshape(-1, 2)

E = esquinas(I_t, N_ESQUINAS)
Mr = cv2.getRotationMatrix2D((W/2, H/2), GIRO, 1.0)
Ir = cv2.warpAffine(I_t, Mr, (W, H), borderMode=cv2.BORDER_REPLICATE)
Er = esquinas(Ir, N_ESQUINAS)
proy = (Mr[:, :2] @ E.T).T + Mr[:, 2]            # dónde caen, en la foto girada, las de la foto de antes
dentro = (proy[:, 0] > 5) & (proy[:, 0] < W - 5) & (proy[:, 1] > 5) & (proy[:, 1] < H - 5)
d = np.linalg.norm(proy[dentro][:, None] - Er[None], axis=2).min(1)
N_DENTRO, REENCONTRADAS = dentro.sum(), (d < 2).sum()   # las que siguen dentro, y las reencontradas a menos de 2 px
Es = esquinas(cv2.resize(I_t, (W//2, H//2), interpolation=cv2.INTER_AREA), N_ESQUINAS)*2 + 0.5   # a la mitad de tamaño
EN_DISCO = (np.hypot(Es[:, 0] - uc, Es[:, 1] - vc) < 25).sum()
# para la figura: la foto a la mitad de tamaño y el mapa de c, con la misma ventana de 7 x 7 que M_est
I_MITAD = cv2.resize(I_t, (W//2, H//2), interpolation=cv2.INTER_AREA)
S_uu, S_vv, S_uv = (cv2.boxFilter(a_, -1, (7, 7), normalize=False) for a_ in (Ix*Ix, Iy*Iy, Ix*Iy))
C_HARRIS = S_uu*S_vv - S_uv**2 - 0.04*(S_uu + S_vv)**2

In [ ]:
# @title ✔ Resultado
for nom, lam, c in SITIOS:
    print(f'{nom:24s} autovalores {lam[0]:9.0f} y {lam[1]:9.0f} ; c = {coma(c, 0)}')
print(f'\n{len(E)} esquinas; girando la foto {GIRO:g}°, de las {N_DENTRO} que siguen dentro se reencuentran {REENCONTRADAS} (a menos de 2 px)')
print(f'A la mitad de tamaño, con la misma ventana: {len(Es)} esquinas, {EN_DISCO} en el disco, que no tiene')

from matplotlib.colors import SymLogNorm
SUP = str.maketrans('0123456789-', '⁰¹²³⁴⁵⁶⁷⁸⁹⁻')
def potencia(c):                                  # c con coma y potencia de 10: −6,5·10⁶
    if abs(c) < 1e4:
        return coma(c, 0)
    e = int(np.floor(np.log10(abs(c))))
    return ('+' if c > 0 else '') + coma(c/10**e, 1) + '·10' + str(e).translate(SUP)

fig, axs = plt.subplots(2, 2, figsize=(12, 9), gridspec_kw=dict(left=0.01, right=0.99, bottom=0.01, top=0.95, wspace=0.04, hspace=0.16))
(a0, a1), (a2, a3) = axs
a0.imshow(I_t, cmap='gray', vmin=0, vmax=255); a0.plot(E[:, 0], E[:, 1], 'o', mfc='none', mec=ROJO, ms=12, mew=2)
a0.set_title(f'Harris: {len(E)} esquinas', fontsize=12)
a1.imshow(Ir, cmap='gray', vmin=0, vmax=255); a1.plot(Er[:, 0], Er[:, 1], 'o', mfc='none', mec=IQS_VERDE, ms=12, mew=2)
a1.set_title(f'Girada {GIRO:g}°: de las {N_DENTRO} que siguen dentro, {REENCONTRADAS} reencontradas', fontsize=12)
a2.imshow(I_MITAD, cmap='gray', vmin=0, vmax=255, extent=(-0.5, W - 0.5, H - 0.5, -0.5))
a2.plot(Es[:, 0], Es[:, 1], 'o', mfc='none', mec=AMBAR, ms=12, mew=2)
a2.set_title(f'A la mitad de tamaño ({W//2} × {H//2}): {len(Es)} esquinas, {EN_DISCO} en el disco', fontsize=12)
a3.imshow(C_HARRIS, cmap='RdBu_r', norm=SymLogNorm(linthresh=1e6, vmin=-1e8, vmax=1e8))
for (nom, lam, c), (u_, v_) in zip(SITIOS, SITIOS_UV):
    a3.plot(u_, v_, 'o', mfc='none', mec='k', ms=12, mew=2)
    a3.text(u_ - 18, v_, nom.replace(' del', '\ndel') + f'\nc = {potencia(c)}', fontsize=10, va='center', ha='right',
            bbox=dict(fc='w', ec='none', alpha=0.85, pad=1.5))
a3.set_title('El mapa de c (escala logarítmica): blanco, plano;\nazul, borde (c < 0); rojo, esquina (c > 0)', fontsize=12)
sin_ejes(a0, a1, a2, a3); plt.show()

**Qué tiene que salir:** c negativa en el borde y positiva en la esquina; girando 30°, se reencuentran las 5 que siguen dentro; a la mitad, 4 en el disco; en la figura, el anillo del disco, azul y sin rojo: no tiene esquinas.

### Ejercicio 1 (en casa): los dos mandos de Canny

Con σ = 1 y los umbrales (1 y 3), (3 y 8) y (8 y 20): ¿cuántos píxeles de borde, y cuántos en la mesa? ¿Qué mando decide la escala y cuál la evidencia?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el suavizado [px]
E1_SIGMA = 1  # @param {type:"number"}
# los pares de umbrales (bajo, alto) que se prueban [niveles por píxel]
E1_UMBRALES = [(1, 3), (3, 8), (8, 20)]  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
e1_J = cv2.GaussianBlur(I_l, (0, 0), E1_SIGMA) if E1_SIGMA else I_l   # la foto con la lámpara, suavizada
E1 = []                                          # (bajo, alto, píxeles de borde, de ellos en la mesa lejos de las piezas)
E1_BORDES = []                                   # para la figura: los bordes de cada par, recortados y engrosados a 2 px
for e1_bajo, e1_alto in E1_UMBRALES:
    e1_b = cv2.Canny(e1_J, e1_bajo*8, e1_alto*8, L2gradient=True)
    E1.append((e1_bajo, e1_alto, int((e1_b > 0).sum()), int((e1_b[lejos] > 0).sum())))
    E1_BORDES.append(cv2.dilate(e1_b[60:480, 100:640], np.ones((2, 2), np.uint8)))

In [ ]:
# @title ✔ Resultado
for e1_bajo, e1_alto, e1_px, e1_mesa in E1:
    print(f'σ = {E1_SIGMA:g}, umbrales {e1_bajo:g} y {e1_alto:g}: {e1_px} px de borde; {e1_mesa} en la mesa, lejos de las piezas')

fig, axs = plt.subplots(1, len(E1), figsize=(15, 3.8), squeeze=False)
for ax, (e1_bajo, e1_alto, e1_px, e1_mesa), e1_b in zip(axs[0], E1, E1_BORDES):
    ax.imshow(255 - e1_b, cmap='gray', vmin=0, vmax=255); sin_ejes(ax)
    ax.set_title(f'umbrales {e1_bajo:g} y {e1_alto:g}: {e1_px} px\n({e1_mesa} en la mesa, lejos de las piezas)', fontsize=12)
fig.suptitle(f'Canny con σ = {E1_SIGMA:g}, con la lámpara (recorte: u de 100 a 639, v de 60 a 479; bordes engrosados a 2 px)', fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.92)); plt.show()

### Ejercicio 2 (en casa): una foto real

`skimage.data.coins()`: 24 monedas reales, con más luz arriba. ¿Cuántas salen enteras con Otsu y con el adaptativo de 51 × 51 y de 101 × 101? ¿Por qué el de 51, que en la mesa iba bien, aquí pierde?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# los umbrales adaptativos que se prueban: (lado del vecindario [px, impar], margen [niveles de gris])
E2_ADAPTATIVOS = [(51, 10), (101, 5)]  # @param {type:"raw"}
# una moneda entera: área de y hasta [px] y circularidad mínima
E2_AREA = (500, 2600)  # @param {type:"raw"}
E2_CIRC = 0.8  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
from skimage import data
E2_FOTO = data.coins()                           # la foto real: 24 monedas sobre una mesa

def e2_cuenta(b):
    """Tras una apertura de 3 x 3: las componentes de E2_AREA[0] px o más y, de ellas, las monedas enteras."""
    b = cv2.morphologyEx(b, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, et, st, _ = cv2.connectedComponentsWithStats(b)
    grandes = [k for k in range(1, n) if st[k, cv2.CC_STAT_AREA] >= E2_AREA[0]]
    enteras = [k for k in grandes if st[k, cv2.CC_STAT_AREA] < E2_AREA[1]
               and piezas_en(((et == k)*255).astype(np.uint8), *E2_AREA)[0]['circ'] > E2_CIRC]
    return len(grandes), len(enteras), b

_, e2_otsu = cv2.threshold(E2_FOTO, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
E2 = [('Otsu', *e2_cuenta(e2_otsu))]            # (método, componentes, monedas enteras, máscara)
for e2_lado, e2_margen in E2_ADAPTATIVOS:
    e2_b = cv2.adaptiveThreshold(E2_FOTO, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, e2_lado, -e2_margen)
    E2.append((f'adaptativo {e2_lado} x {e2_lado}, más {e2_margen}', *e2_cuenta(e2_b)))

In [ ]:
# @title ✔ Resultado
for e2_nom, e2_n, e2_ent, _ in E2:
    print(f'{e2_nom}: {e2_n} componentes de {E2_AREA[0]} px o más; {e2_ent} monedas enteras')

fig, axs = plt.subplots(1, 1 + len(E2), figsize=(15, 3.2))
axs[0].imshow(E2_FOTO, cmap='gray'); axs[0].set_title('La foto')
for ax, (e2_nom, _, e2_ent, e2_m) in zip(axs[1:], E2):
    ax.imshow(e2_m, cmap='gray'); ax.set_title(e2_nom.replace(' x ', ' × ') + f': {e2_ent} enteras', fontsize=10)
sin_ejes(*axs); plt.tight_layout(); plt.show()

### Ejercicio 3 (en casa): la pieza en la esquina del campo

Con la cámara del curso y el disco en (0,65; 0,45) m, cerca de la esquina: ¿a cuántos milímetros sale con la cara, con la mesa y sin quitar la distorsión? ¿Qué error crece hacia el borde y cuál no?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# dónde se pone el disco, en la mesa [m]
E3_POS = (0.65, 0.45)  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
e3_cam = cam if EQUIPO == 0 else Mesa()          # la cámara del curso, en el plano del montaje, sea cual sea EQUIPO
E3_CAL = (K_CURSO, DIST_CURSO, T_C0_verdad())     # y su calibración, exacta
I3, V3 = e3_cam.foto([Pieza('disco', E3_POS[0], E3_POS[1], 0.040)], mascara=True)
_, b3 = cv2.threshold(I3, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
p3 = piezas_en(b3)[0]                            # el disco, en la imagen
E3 = []                                          # (cómo, a cuánto de su sitio [mm])
E3_PUNTOS = []                                   # para la figura: dónde sale, en la mesa [m]
for e3_nom, e3_dist, e3_z in (('cortando con la cara', E3_CAL[1], H_PIEZA), ('cortando con la mesa', E3_CAL[1], 0.0),
                              ('sin quitar la distorsión', np.zeros(5), H_PIEZA)):
    e3_Q = a_la_mesa((p3['u'], p3['v']), E3_CAL[0], e3_dist, E3_CAL[2], z=e3_z)[0]
    E3.append((e3_nom, 1e3*np.linalg.norm(e3_Q - np.array(E3_POS))))
    E3_PUNTOS.append(e3_Q)
E3_EJE = 1e3*np.linalg.norm(np.array(E3_POS) - TC[:2])   # a cuánto del eje óptico [mm]

In [ ]:
# @title ✔ Resultado
print(f'El disco en ({coma(E3_POS[0])}; {coma(E3_POS[1])}) m, a {coma(E3_EJE, 0)} mm del eje óptico, en el píxel ({coma(p3["u"], 1)}; {coma(p3["v"], 1)}):')
for e3_nom, e3_mm in E3:
    print(f'  {e3_nom}: a {coma(e3_mm)} mm de su sitio')

from matplotlib.patches import Circle
fig, axs = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw=dict(width_ratios=[1.3, 1]))
ax = axs[0]
ax.imshow(I3, cmap='gray', vmin=0, vmax=255)
ax.plot([E3_CAL[0][0, 2], p3['u']], [E3_CAL[0][1, 2], p3['v']], '--', color='w', lw=1.2)
ax.plot(E3_CAL[0][0, 2], E3_CAL[0][1, 2], 'o', color='w', ms=6)
ax.text(E3_CAL[0][0, 2] + 8, E3_CAL[0][1, 2] + 22, 'el eje óptico', color='w', fontsize=11)
ax.plot(p3['u'], p3['v'], '+', color='k', ms=14, mew=2)
ax.set_xlabel('u [px]'); ax.set_ylabel('v [px]')
ax.set_title(f'La foto: el disco en ({coma(E3_POS[0])}; {coma(E3_POS[1])}) m, a {coma(E3_EJE, 0)} mm del eje óptico', fontsize=12)
ax = axs[1]                                       # en la mesa, alrededor de su sitio
e3_c = 1e3*np.array(E3_POS)
ax.add_patch(Circle(e3_c, 1e3*D_PIEZA/2, fc='none', ec=IQS_VERDE, lw=1.5))
ax.plot(*e3_c, '+', color=IQS_VERDE, ms=18, mew=2.5, label='su sitio (y el disco)')
for (e3_nom, e3_mm), e3_Q, (mk, col) in zip(E3, E3_PUNTOS, (('o', IQS_AZUL), ('^', AMBAR), ('x', ROJO))):
    ax.plot(*1e3*e3_Q, mk, color=col, ms=9, mew=2.5, label=f'{e3_nom}: {coma(e3_mm)} mm')
e3_L = max(25, 1.25*max(e3_mm for _, e3_mm in E3))
if E3_EJE > 1:                                    # la recta hacia el eje óptico (en el eje no hay hacia dónde)
    e3_dir = (TC[:2] - np.array(E3_POS))/np.linalg.norm(TC[:2] - np.array(E3_POS))
    ax.plot(*np.array([e3_c - 1.5*e3_L*e3_dir, e3_c + 1.5*e3_L*e3_dir]).T, '--', color='k', lw=0.8)
    ax.annotate('hacia el eje óptico', e3_c + 0.95*e3_L*e3_dir, e3_c + 0.55*e3_L*e3_dir + 0.25*e3_L*np.array([e3_dir[1], -e3_dir[0]]),
                fontsize=10, ha='center', va='center', arrowprops=dict(arrowstyle='->', color='k'))
ax.set_xlim(e3_c[0] - e3_L, e3_c[0] + e3_L); ax.set_ylim(e3_c[1] - e3_L, e3_c[1] + e3_L); ax.set_aspect('equal')
ax.grid(True, color='#E6E6E6'); ejes(ax); ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.14), fontsize=10, frameon=False, ncol=2)
ax.set_title('En la mesa, desde arriba (mm)', fontsize=12)
plt.tight_layout(); plt.show()

---

## Soluciones

**Ejercicio de clase.** Con la cámara del curso y la lámpara: con Otsu (68), el disco no aparece; se une a la mitad clara de la mesa. Con el 144, sale cortado por el lado oscuro (754 px frente a 813): el centroide se va 0,21 px y la pieza, a 0,25 mm de P, unas ocho veces lo que tiembla el centroide (0,03 px, unas 0,03 mm); es sistemático y promediando fotos no se quita. Con el adaptativo, a 0,03 mm, casi como con el techo (0,02). Con una calibración de S16, a menos de 0,5 mm (de 0,06 a 0,20 con la tabla mejorada; de 0,23 a 0,45 con la de partida); sin calibrar, de 9 a 18 mm: la unidad no es la hoja de datos ni está donde dice el plano. Con `EQUIPO = 0`, «sin calibrar» solo quita la distorsión: 0,27 mm.

**Ejercicio 1.** Con σ = 1: (1 y 3), 44 348 px de borde y 42 695 en la mesa; (3 y 8), 560 y 0; (8 y 20), 366 y 0. σ decide la escala, qué detalle cuenta como borde; los umbrales, cuánta evidencia se exige: bajarlos deja pasar la veta, y subirlos pierde los bordes débiles, como el de la sombra. Subir los umbrales cuando el problema era la escala es el error habitual.

**Ejercicio 2.** Otsu: 25 componentes y 14 monedas enteras; arriba, con más luz, el fondo se une a las monedas de la primera fila. El adaptativo de 51 × 51: 24 y solo 9 enteras, porque el vecindario es del tamaño de una moneda (unos 40 px) y su centro se agujerea. El de 101 × 101 gana: 19. El vecindario tiene que ser bastante mayor que la pieza, como en la mesa (51 px frente a discos de 32).

**Ejercicio 3.** A 430 mm del eje óptico: cortando con la cara, 0,09 mm; con la mesa, 4,26 mm (430·10/990 = 4,3 mm, la cuenta de S16); sin quitar la distorsión, 19,45 mm. El de la cara no crece; los dos sistemáticos sí: el del espesor, en línea recta con la distancia al centro, y el de la distorsión, mucho más deprisa (con r³). En P, sin quitar la distorsión, eran 0,27 mm.

---

## Para llevarse de esta sesión

- **Una imagen es una matriz de números;** el procesado clásico son cuentas con ellos.
- **El umbral global se rompe con la luz;** el adaptativo compara cada píxel con su entorno.
- **El centroide es un píxel:** S16 lo lleva a la mesa y S10, al robot; pesa lo sistemático, no el ruido.
- **Suavizar suma 1 y derivar suma 0;** la sombra también da borde.
- **σ decide la escala de Canny; los umbrales, la evidencia.**

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*